<a href="https://colab.research.google.com/github/Sakinat-Folorunso/OOU_CSC309_Artificial_Intelligence/blob/main/notebooks/CSC309_Week06_Semantic_Networks_Frames_Student_Centred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 309 – Artificial Intelligence
**Week 6 Practical (PH):** Semantic networks and frames — inheritance, defaults and procedural attachment

**Instructor:** Dr Sakinat Folorunso · Associate Professor of AI Systems · Department of Computer Sciences, Olabisi Onabanjo University (OOU), Ago-Iwoye, Ogun State, Nigeria  
**Course:** CSC 309 (2 units; LH 15, PH 45) · **Mode:** student-centred, hands-on, Google Colab  
**CCMAS learning outcomes covered:** LO6

> 🎓 **Instructor note** – this notebook is the *STUDENT (TODO cells are stubs)* version.


## How to use this notebook
1. Fill in the **Group Log**, then do the **Do Now** with your partner.
2. Run the **Setup** cell once (*Runtime → Run all* is safe: nothing here crashes if a TODO is still empty).
3. Work top to bottom. Only edit cells marked **`# TODO(Student)`**.
4. Each task ends with an **auto-check** (✅/❌). Keep going until it is ✅.
5. Finish with the **Reflection**. Fast finishers: try the **Extensions**.

6. **Read the comments.** Every line of code has a plain-English comment above it. Read the comment first, predict what the line does, then run the cell and check your prediction.
7. Hand in the **Assignment** (marks and deadline are in the notebook) after the lab.

*Every random draw uses `SEED = 42`, so your numbers should match your classmates' numbers.*


## Lab tasks at a glance

| Part | Lab task(s) you must complete |
|---|---|
| Part A — Semantic network with inheritance | Task A2 — Extend the network (≥ 2 new concepts, ≥ 2 inheritance queries) |
| Part B — Frames: defaults, instances and demons | Task B1 — Design an `Assessment` frame (CA2 planner) |
| Part C — Which representation when? | *Guided lab: run every cell, read the comments, explain the output in your own words* |
| **Assignment** | see **📝 Assignment / deliverables** near the end |
| **Reflection** | short written reflection at the end (5–7 sentences) |
| *Extensions* | optional, for fast finishers |

In [ ]:
#@title 🧑🏽‍🤝‍🧑🏾 Group Log (fill before you start)
# Form field: type your group members' names here
group_members = "Type names here"   #@param {type:"string"}
# Form field: assign roles (driver, navigator, recorder, presenter)
roles_notes   = "Driver / Navigator / Recorder / Presenter"  #@param {type:"string"}
# Print them so your notebook records who did what
print("👥 Group:", group_members); print("📝 Roles:", roles_notes)


In [ ]:
#@title 🔧 Setup (run once) — seed, logging, helpers
# Standard-library modules used by every lab (os, re, math, random, logging, time ...)
import sys, os, re, math, random, logging, subprocess, warnings, time, itertools
# NumPy: fast arrays and random numbers
import numpy as np
# One seed for the whole course so every run gives the same numbers
SEED = 42                                    # one seed for the whole course → reproducible results
# Seed both Python's and NumPy's random generators
random.seed(SEED); np.random.seed(SEED)
# Configure logging so messages print as "LEVEL | message"
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
# A named logger for this course
log = logging.getLogger("csc309")
# Hide noisy library warnings so output stays readable
warnings.filterwarnings("ignore")

def ensure(pip_name, import_name=None):
    """Import a package; pip-install it first if it is missing (Colab-friendly)."""
    # Try to import the library
    try:
        # Return the imported module
        return __import__(import_name or pip_name)
    # If it is not installed ...
    except ImportError:
        # ... tell the student we are installing it
        log.info("installing %s ...", pip_name)
        # ... and install it with pip
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        # Return the imported module
        return __import__(import_name or pip_name)

# List that stores every check result
_CHECKS = []
def check(name, condition, hint=""):
    """Auto-check. Never raises, so 'Run all' always finishes. Pass a lambda to catch your own bugs."""
    # Try to import the library
    try:
        # Run the check: call it if it is a function, otherwise use the value
        ok = bool(condition() if callable(condition) else condition)
        # Default: no error message
        err = ""
    # If the student's code crashes, we report instead of stopping the notebook
    except Exception as e:                    # student code may crash; report instead of stopping
        # Mark as failed and keep the error message
        ok, err = False, f"{type(e).__name__}: {e}"
    # Store the result
    _CHECKS.append((name, ok))
    # Print a ✅ or ❌ line with a hint on failure
    print(("✅ PASS       " if ok else "❌ TRY AGAIN  ") + name + ("" if ok else f"   → {err or hint}"))
    # Return whether the check passed
    return ok

# Print how many checks passed
def checks_summary():
    # Count the passed checks
    passed = sum(ok for _, ok in _CHECKS)
    # Print the total
    print(f"\nChecks passed: {passed}/{len(_CHECKS)}")

def mean_ci(x, conf=0.95):
    """Mean, sample std and a two-sided t-based confidence interval."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert to a float array and count the observations
    x = np.asarray(x, dtype=float); n = len(x)
    # Mean and sample standard deviation (n-1 divisor)
    m = x.mean(); s = x.std(ddof=1) if n > 1 else 0.0
    # Half-width of the 95% confidence interval from the t-distribution
    h = stats.t.ppf((1 + conf) / 2, n - 1) * s / math.sqrt(n) if n > 1 else 0.0
    # Return mean, SD, lower and upper confidence limits
    return m, s, m - h, m + h

# Print a one-line statistical summary
def report(label, x):
    # Compute the statistics
    m, s, lo, hi = mean_ci(x)
    # Print mean, SD, CI and n
    print(f"{label:<28} mean={m:8.2f}  sd={s:7.2f}  95% CI=[{lo:.2f}, {hi:.2f}]  (n={len(x)})")

def paired_wilcoxon(a, b):
    """Two-sided Wilcoxon signed-rank test for paired samples (same seeds/instances)."""
    # SciPy for the t-distribution
    from scipy import stats
    # Convert both samples to arrays
    a, b = np.asarray(a, float), np.asarray(b, float)
    # If the two samples are identical the test is undefined
    if np.allclose(a, b):
        # Report p = 1 (no difference)
        return 1.0
    # Paired Wilcoxon signed-rank test p-value
    return float(stats.wilcoxon(a, b).pvalue)

# Setup finished
print("✅ Setup complete — seed =", SEED)


### Learning objectives (with Bloom level)
| # | By the end of this lab you can… | Bloom | CCMAS LO |
|---|---|---|---|
| 1 | build a **semantic network** with typed relations (*is-a, instance-of, part-of, property*) | Apply | LO6 |
| 2 | answer queries by **inheritance**, with **exceptions** and detect **conflicts** (multiple inheritance) | Apply / Analyse | LO6 |
| 3 | design **frames** with slots, **defaults** and **demons** (*if-needed*, *if-added*) | Apply / Create | LO6 |
| 4 | choose between network, frame and logic representations for a given task | Evaluate | LO6 |

**Key ideas.** A **semantic network** is a labelled directed graph: nodes are concepts/individuals, edges are relations. Typical relations: **is-a** (subclass ⊂ class), **instance-of** (individual ∈ class), **part-of** (meronymy), **has-property**. A **frame** (Minsky, 1974) is a structured record for a stereotyped situation: a *name*, an **is-a** link and **slots** with facets — *value*, *default*, *if-needed* (compute on demand), *if-added* (trigger when filled).

Logic view: `is-a(Penguin, Bird)` ≡ $\forall x\,(\text{Penguin}(x)\rightarrow\text{Bird}(x))$; a property default is a *defeasible* rule (Week 5, Part D).

**Do Now (3 min).** Draw a 6-node network for *Ada, Student, Person, OOU, Faculty of Science, Computer Sciences* using at least three different relation labels.

## Part A — Semantic network with inheritance
**Inheritance rule.** To find property $p$ of node $n$: use the value stored at $n$; otherwise look at its parents, *nearest first*. A value at a more specific class **overrides** the general one (exceptions). If two parents *at the same distance* disagree, report a **conflict** (the Nixon diamond).

In [ ]:
# TODO(Student): finish SemNet.ancestors() and SemNet.get()
# defaultdict creates missing dictionary entries automatically
from collections import defaultdict

# A semantic network: nodes joined by labelled links, with properties on nodes
class SemNet:
    # Constructor
    def __init__(self):
        # All links stored as (node a, relation, node b)
        self.links = []                      # (a, relation, b)
        # Properties of each node: node -> {property: value}
        self.props = defaultdict(dict)       # node → {property: value}
    # Add a link; returning self lets us chain several add_link calls
    def add_link(self, a, rel, b):  self.links.append((a, rel, b)); return self
    # Attach a property value to a node (also chainable)
    def add_prop(self, node, prop, value): self.props[node][prop] = value; return self

    # Direct parents of a node through is-a / instance-of links
    def parents(self, node):
        # Collect b for every link a --is-a--> b where a is our node
        return [b for a, r, b in self.links if a == node and r in ("is-a", "instance-of")]

    def ancestors(self, node):
        """All classes reachable via is-a / instance-of (transitive closure), nearest first."""
        # Placeholder so the cell runs; replace with your list of ancestors
        return []

    def get(self, node, prop):
        """Value of `prop` for `node` via inheritance (nearest wins). Return 'CONFLICT' if equally-near parents disagree; None if unknown."""
        # Placeholder; replace with your property lookup
        return None

In [ ]:
#@title 🐧 Build two networks (animals + OOU) and query them
# Create an empty network
net = SemNet()
# Taxonomy: Bird and Mammal are Animals; Penguin is a Bird
net.add_link("Bird", "is-a", "Animal").add_link("Mammal", "is-a", "Animal").add_link("Penguin", "is-a", "Bird")
# More classes and an instance (Opus is a penguin)
net.add_link("Sparrow", "is-a", "Bird").add_link("Human", "is-a", "Mammal").add_link("Opus", "instance-of", "Penguin")
# The Nixon diamond: Nixon is both a Quaker and a Republican
net.add_link("Quaker", "is-a", "Person").add_link("Republican", "is-a", "Person").add_link("Nixon", "instance-of", "Quaker").add_link("Nixon", "instance-of", "Republican")
# Properties: animals move, birds fly and have wings
net.add_prop("Animal", "can_move", True).add_prop("Bird", "can_fly", True).add_prop("Bird", "has_wings", True)
# Exception: penguins cannot fly; humans use language
net.add_prop("Penguin", "can_fly", False).add_prop("Human", "uses_language", True)
# Conflicting inherited property: Quakers are pacifists, Republicans are not
net.add_prop("Quaker", "pacifist", True).add_prop("Republican", "pacifist", False)

# part-of hierarchy for OOU (meronymy is NOT inherited like is-a)
# Part-of links for the OOU organisation
net.add_link("Computer Sciences", "part-of", "Faculty of Science").add_link("Faculty of Science", "part-of", "OOU")
# A student hierarchy for Ada
net.add_link("Ada", "instance-of", "UndergraduateStudent").add_link("UndergraduateStudent", "is-a", "Student").add_link("Student", "is-a", "Person")

# Follow part-of links upward
def part_of_chain(net, x):
    # Start the chain with the node itself
    chain = [x]
    # Repeat until there is no larger part
    while True:
        # Find the part this node belongs to
        nxt = [b for a, r, b in net.links if a == chain[-1] and r == "part-of"]
        # No more parts: return the chain
        if not nxt: return chain
        # Add the next level
        chain.append(nxt[0])

# Questions to ask the network
queries = [("Opus", "can_fly"), ("Sparrow", "can_fly"), ("Opus", "has_wings"), ("Opus", "can_move"), ("Human", "has_wings"), ("Nixon", "pacifist")]
# Print each answer by inheritance
for n, p in queries: print(f"{n:>8}.{p:<14} → {net.get(n, p)}")
# Print ancestors and the part-of chain
print("Ancestors of Opus:", net.ancestors("Opus")); print("Part-of chain:", " ⊂ ".join(part_of_chain(net, "Computer Sciences")))
check("Penguins can't fly (exception overrides Bird default)", lambda: net.get("Opus", "can_fly") is False)
check("Sparrows fly (inherited from Bird, 1 level)",           lambda: net.get("Sparrow", "can_fly") is True)
check("Opus has wings & can move (inherited 2–3 levels)",       lambda: net.get("Opus", "has_wings") is True and net.get("Opus", "can_move") is True)
check("unknown property → None",                                lambda: net.get("Human", "has_wings") is None)
check("Nixon diamond → CONFLICT",                               lambda: net.get("Nixon", "pacifist") == "CONFLICT")
check("Ada is a Person (transitive is-a)",                      lambda: "Person" in net.ancestors("Ada"))

### Task A2 — Extend the network (≥ 2 new concepts, ≥ 2 inheritance queries)
Add to `net` at least **two** new classes with **one exception** (e.g. `Ostrich` is-a `Bird` with `can_fly=False`; `Bat` is-a `Mammal` with `can_fly=True`). Then print two queries that prove inheritance and override.

**Visualise** the network below, then decide: which relation would you have to *forbid* from inheriting (part-of!) and why? (*Is Ada part-of the Faculty of Science if Computer Sciences is?* — No: part-of is **not** transitive through instance-of.)

In [ ]:
#@title 🖼️ Draw the network
# networkx draws graphs
import networkx as nx
# Plotting library
import matplotlib.pyplot as plt
# Draw a semantic network as a graph
def draw_net(net, title="Semantic network"):
    # Create a directed graph
    G = nx.DiGraph()
    # One edge per link, labelled with the relation
    for a, r, b in net.links: G.add_edge(a, b, label=r)
    # Layout with a fixed seed so the picture is reproducible
    pos = nx.spring_layout(G, seed=SEED, k=0.9)
    # Draw nodes and arrows
    plt.figure(figsize=(8, 5.5)); nx.draw_networkx(G, pos, node_size=1300, node_color="#dbeafe", font_size=8, arrows=True)
    # Write the relation names on the edges
    nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, "label"), font_size=7)
    # Title, hide axes, display
    plt.title(title); plt.axis("off"); plt.tight_layout(); plt.show()
# Draw the network we built
draw_net(net)

## Part B — Frames: defaults, instances and demons
Frame lookup order for slot *s* of an **instance**: (1) a local *value*; (2) an *if-needed* demon found up the chain, run on the instance; (3) the nearest *default*. Setting a slot fires the nearest *if-added* demon.

In [ ]:
#@title 📦 A small frame system
# A frame = a named object with slots, inheriting from a parent frame
class Frame:
    # Constructor: name, parent (isa), and slot values
    def __init__(self, name, isa=None, **slots):
        # Store the name, parent and an empty slot table
        self.name, self.isa, self.slots = name, isa, {}
        # Each slot is a dict of facets (value, default, if_needed, if_added)
        for k, v in slots.items(): self.slots[k] = v if isinstance(v, dict) else {"value": v}   # allow facets: dict(default=…, if_needed=…)

    # Generator that walks up the inheritance chain
    def chain(self):
        # Start at this frame
        f = self
        # Yield it, then move to the parent
        while f is not None: yield f; f = f.isa

    # Look up one facet of a slot, nearest frame first
    def _facet(self, slot, facet):
        # Walk up the chain
        for f in self.chain():
            # Return the first frame that defines this facet
            if facet in f.slots.get(slot, {}): return f.slots[slot][facet]
        # Placeholder; replace with your property lookup
        return None

    # Get the value of a slot
    def get(self, slot):
        # 1) A local value wins
        v = self._facet(slot, "value")
        # Return it if present
        if v is not None: return v
        # 2) Otherwise an if-needed demon (a function that computes it)
        demon = self._facet(slot, "if_needed")
        # Run the demon on this frame
        if demon is not None: return demon(self)                 # computed on demand from other slots
        # 3) Otherwise the nearest default
        return self._facet(slot, "default")

    # Set a slot value
    def set(self, slot, value):
        # Store the value
        self.slots.setdefault(slot, {})["value"] = value
        # Look for an if-added demon (a trigger)
        demon = self._facet(slot, "if_added")
        # Run the trigger when the value is set
        if demon is not None: demon(self, value)                 # trigger: e.g. set a flag
        # Allow chaining
        return self

    # Create an instance (child frame)
    def instantiate(self, name, **values):
        # The child inherits from this frame
        child = Frame(name, isa=self)
        # Set the given slot values (this fires triggers)
        for k, v in values.items(): child.set(k, v)
        # Return the new instance
        return child

    # Show all slots of a frame
    def describe(self):
        # Collect every slot name along the chain
        keys = sorted({k for f in self.chain() for k in f.slots})
        # Resolve each slot's value
        return {k: self.get(k) for k in keys}

def grade_class(f):
    """Nigerian 5-point scale (NUC): First 4.50–5.00, 2:1 3.50–4.49, 2:2 2.40–3.49, Third 1.50–2.39, Pass 1.00–1.49."""
    # Read the CGPA of a student frame
    g = f.get("cgpa")
    # No CGPA: no class of degree
    if g is None: return None
    # Nigerian 5-point scale: map CGPA to a degree class
    return ("First Class" if g >= 4.5 else "Second Class Upper" if g >= 3.5 else
            "Second Class Lower" if g >= 2.4 else "Third Class" if g >= 1.5 else "Pass" if g >= 1.0 else "Fail")

# A general Person frame with default values
Person  = Frame("Person", species="human", status="active")
# Student frame inherits from Person and sets defaults
Student = Frame("Student", isa=Person, level={"default": 100}, cgpa={"value": None},
                class_of_degree={"if_needed": grade_class},
                standing={"if_needed": lambda f: None})
# Trigger: flag students at risk of probation
def flag_probation(f, v):                                         # if-added demon on cgpa
    # Set the standing slot depending on the CGPA
    f.slots["standing"] = {"value": "probation-risk" if v < 1.5 else "good"}
# Attach the trigger to the cgpa slot
Student.slots["cgpa"]["if_added"] = flag_probation

# Ada: a strong student
ada = Student.instantiate("Ada", level=300, cgpa=4.62)
# Tayo: a student at risk
tayo = Student.instantiate("Tayo", cgpa=1.2)
# Show both frames
print("Ada :", ada.describe()); print("Tayo:", tayo.describe())
check("Ada inherits species=human from Person",            lambda: ada.get("species") == "human")
check("Tayo uses the DEFAULT level 100; Ada's own value 300", lambda: tayo.get("level") == 100 and ada.get("level") == 300)
check("if-needed demon computes First Class from cgpa",       lambda: ada.get("class_of_degree") == "First Class")
check("if-added demon flagged Tayo's low cgpa",               lambda: tayo.get("standing") == "probation-risk")

### Task B1 — Design an `Assessment` frame (CA2 planner)
Create a frame `Assessment` with slots `max_mark` (default 10) and an **if-needed** slot `percent` $=100\times\text{score}/\text{max\_mark}$. Create a child frame `CA2` for your group's plan (e.g. `score=7.5`). Add a **demon** `if_added` on `score` that stores `"low"` in slot `flag` when `score < 5`. Finally, instantiate a `CSC309` course frame whose `total_CA` is **if-needed**: the sum of three assessment frames' scores (CA1, CA2, CA3).

In [ ]:
# TODO(Student): build the frames described above
# TODO: build an Assessment frame
Assessment = None      # Frame("Assessment", max_mark={"default": 10}, percent={"if_needed": ...})
# TODO: three assessment instances
ca1 = ca2 = ca3 = None # instances with scores 8, 7.5 and 4
# TODO: a course frame that sums CA marks on demand
CSC309 = None          # Frame("Course", total_CA={"if_needed": lambda f: sum(a.get("score") for a in ...)})

In [ ]:
#@title ✅ Auto-check for Task B1
check("max_mark default (10) inherited by CA2",           lambda: ca2.get("max_mark") == 10)
check("CA2 percent = 75 (if-needed demon)",               lambda: abs(ca2.get("percent") - 75.0) < 1e-9)
check("if-added demon flags CA3 (4 < 5) as 'low'",        lambda: ca3.get("flag") == "low" and ca1.get("flag") == "ok")
check("total_CA = 19.5 computed on demand",               lambda: abs(CSC309.get("total_CA") - 19.5) < 1e-9)

### Worked demo — the class note's weight demon (Examples 6.3 and 6.4)
An **if-needed** demon can compute $w_i=\max_i/\sum_j\max_j$ on demand, and an **if-added** demon can protect a frame from bad data (a score above `max_mark`). Read the code, run it, then change the maximum marks and predict the weights before you run again.

**Strict links vs defaults.** `is-a` and `instance-of` links translate directly into logic ($\forall x\,(\mathrm{Penguin}(x)\rightarrow\mathrm{Bird}(x))$). *Default* inheritance ("birds fly") does **not**: it is *non-monotonic*, because learning that Tweety is a penguin withdraws the conclusion that Tweety flies. In logic you would need an abnormality predicate, $\mathrm{Bird}(x)\wedge\neg\mathrm{Ab}(x)\rightarrow\mathrm{Flies}(x)$ (Week 5, Part D).

In [ ]:
#@title ⚖️ Weights demon + validation demon (note Examples 6.3 / 6.4)
# Components of the assessment
parts = [Frame("test", max_mark=20), Frame("assignment", max_mark=10), Frame("project", max_mark=20)]
# Course frame with weights
Course = Frame("Course", parts=parts,
               weights={"if_needed": lambda f: {p.name: p.get("max_mark") / sum(q.get("max_mark") for q in f.get("parts")) for p in f.get("parts")}})
# Print the weights
print("weights:", Course.get("weights"))

# Trigger that validates marks
def reject_too_high(f, v):
    # Raise an error when the score exceeds the maximum
    if v > f.get("max_mark"): raise ValueError(f"score {v} exceeds max_mark {f.get('max_mark')}")
# A frame that uses the validator
Checked = Frame("CheckedAssessment", max_mark=20, score={"if_added": reject_too_high})
# Try to enter an invalid score
try:
    # Remember that it was accepted (it should not be)
    Checked.instantiate("T1", score=25); rejected = False
# Catch the error
except ValueError as e:
    # Mark as rejected and print the message
    rejected = True; print("if-added demon says:", e)
# Read the weights again for a check
w = Course.get("weights")
check("weights are 0.4, 0.2, 0.4 and sum to 1", lambda: abs(w["test"] - 0.4) < 1e-9 and abs(w["assignment"] - 0.2) < 1e-9 and abs(sum(w.values()) - 1) < 1e-9)
check("Exercise 6.3: (30, 10, 10) → 0.6, 0.2, 0.2", lambda: [round(x, 1) for x in (30/50, 10/50, 10/50)] == [0.6, 0.2, 0.2])
check("if-added demon rejects score 25 > 20", lambda: rejected)

## Part C — Which representation when?
| | Semantic network | Frame | Predicate logic |
|---|---|---|---|
| Strength | taxonomies, associations, spreading search | stereotypes, **defaults**, procedural attachment | precise semantics, sound inference |
| Weakness | no standard semantics; exceptions ad hoc | no quantifiers; hard to reason *across* frames | brittle with exceptions; computationally costly |
| Typical use | WordNet, knowledge graphs | expert-system shells, OOP, ontologies | theorem proving, verification |

**Exercise.** Write the logic form of: *every undergraduate student is a student*, *students normally start at level 100* (default!), and explain why the second cannot be a plain FOL implication.

## 📝 Assignment (5 marks, individual, formative)
1. *(2)* Build a **semantic network of at least 10 nodes** on a domain you know (for example Nigerian music genres), including one *exception* and one *conflict*, and query it.
2. *(2)* Build a frame system with one *if-needed* and one *if-added* demon; show a trace of each demon firing.
3. *(1)* Write a table that says when you would use a semantic network, a frame system or logic, with one example each.

*Marked with the course rubric (class note, Table 0.3): correctness, evidence (numbers with CI and a named test), code clarity, interpretation. Submit the **executed notebook** (Runtime → Run all, outputs visible) as a Colab share link or `.ipynb` before the next lab.*

## Reflection
- How are *frames* related to classes and objects in Python? What do they have that a Python class lacks (demons? defaults with exceptions?)?
- Give an example where *multiple inheritance* causes a conflict in a real knowledge base. How would you resolve it?

```text
Write your reflection here…
```

## Extensions
- Add **spreading activation**: activate "Penguin" and spread energy along links with decay 0.5; which concepts light up?
- Export your network to RDF-style triples and run a query "everything that is-a Animal".

In [ ]:
checks_summary()  # how many auto-checks have you passed?